# AME 5003 — Principles of NLP
## Lab 1: From Raw Text to a Working Text-Processing Pipeline

**Manipal School of Information Sciences (MSIS)**  
**Manipal Academy of Higher Education (MAHE)**

### What you will build
By the end of this lab, you will build a small processor for realistic student-service messages that can:

- extract structured fields using regular expressions;
- tokenize text and inspect boundary errors;
- normalize selected surface variations;
- make task-aware stop-word decisions;
- compare stemming and lemmatization;
- combine these steps into an end-to-end text-processing pipeline.

### Lab method
**Observe → Predict → Try → Test → Break → Improve → Explain**

You may use documentation, web search, Stack Overflow, or AI tools. However, for every major task you must:
1. write your prediction/approach first;
2. test the solution on the supplied examples;
3. find at least one case where it fails;
4. explain one assumption made by your solution.

> **Important:** The goal is not merely to make code run. The goal is to understand what information each processing decision preserves or destroys.


> **Dataset note:** The MSIS/MAHE-style messages, IDs and email addresses used here are illustrative and fictional; they are designed only for teaching text-processing decisions.


## Part 0 — Before code: what is the computer supposed to see?
**Suggested time: 0–15 min**

Read the following message:

> **URGENT!!!** Fee of **Rs. 2,450** was paid on **21/08/2026**.  
> The amount is **not reflected yet**.  
> Ref: **MSIS/26/104**  
> Contact: **student23@learner.manipal.edu**

### Activity 0.1 — Human extraction
Without writing code, identify:

| Field | Your answer |
|---|---|
| Urgency |  |
| Amount |  |
| Date |  |
| Reference ID |  |
| Email |  |
| Complaint meaning / intent |  |

### Activity 0.2 — Method choice
For each field above, decide whether you would begin with:

- **Regex / pattern rules**
- **Language-aware NLP / ML**
- **A hybrid of both**

Write one sentence explaining your choice.

### Checkpoint
Be ready to answer:

> **What kind of information is naturally represented by stable character patterns, and what kind depends on meaning/context?**


### Environment note
The notebook uses only lightweight CPU-friendly tools. Python's built-in `re` module is enough for the first half. Later sections use NLTK. In Google Colab NLTK is usually available; if your environment does not have it, install it with `pip install nltk`.

### Load the common lab dataset

In [1]:

messages = [
    "URGENT!!! Fee of Rs. 2,450 was paid on 21/08/2026. The amount is not reflected yet. Ref: MSIS/26/104. Contact: student23@learner.manipal.edu",
    "Payment received: ₹3500 on 20-08-2026. Ref MSIS/FIN/2108. Mail: accounts.msis@manipal.edu",
    "Fee INR 12750 paid yesterday. Transaction ID: MAHE/PG/778. Email: student.one@gmail.com",
    "I have NOT received my refund of INR 2450. Please contact me at user24@learner.manipal.edu",
    "Registration completed successfully. Ref: MSIS/REG/4412",
    "My exam fee of Rs.2450 is still pending even though I paid on 19/08/2026.",
    "Approx amount: two thousand rupees. Payment is not showing.",
    "Hostel payment ₹2,450.50 received. Ref: MAHE/HST/219.",
    "Fee payment ₹2.5k is pending.",
    "Please update attendance for NLP. Student ID: MSISPG2026-041.",
    "Meeting rescheduled to 3.30 p.m. on 22/08/2026. Contact: faculty.msis@manipal.edu",
    "Good service!!! Really GOOD. Refund was processed quickly."
]

print("Number of messages:", len(messages))
for i, m in enumerate(messages[:3], 1):
    print(f"\n{i}. {m}")


Number of messages: 12

1. URGENT!!! Fee of Rs. 2,450 was paid on 21/08/2026. The amount is not reflected yet. Ref: MSIS/26/104. Contact: student23@learner.manipal.edu

2. Payment received: ₹3500 on 20-08-2026. Ref MSIS/FIN/2108. Mail: accounts.msis@manipal.edu

3. Fee INR 12750 paid yesterday. Transaction ID: MAHE/PG/778. Email: student.one@gmail.com


## Part 1 — Regex from scratch
**Suggested time: 15–45 min**

Regular expressions describe **surface patterns** in text.

### Activity 1.1 — Predict before running
For each pattern below, write what you expect it to match:

- `r"\d"`
- `r"\d+"`
- `r"\d{2}-\d{2}-\d{4}"`
- `r"[A-Z]+"`

Then test your prediction on one message from `messages`.

### Activity 1.2 — Build a date extractor
Your extractor should first handle:

- `21-08-2026`
- `21/08/2026`

Then test whether it handles:

- `21-08-26`
- `3.30 p.m.`
- `99-99-2026`

Before changing your pattern, write:

> **What date formats does my application actually need to accept?**

### TODO
Create `date_pattern` and test it.


In [2]:

import re

sample = messages[0]

# TODO 1.1: try the four simple patterns from the activity.
# Example structure:
# pattern = r""
# print(re.findall(pattern, sample))


# TODO 1.2: write a date pattern for DD-MM-YYYY and DD/MM/YYYY.
date_pattern = r"\d{2}(-|/)\d{2}(-|/)\d{2}+"

date_tests = [
    "Date: 21-08-2026",
    "Date: 21/08/2026",
    "Date: 21-08-26",
    "Invalid-looking date: 99-99-2026",
]

for text in date_tests:
    match = re.search(date_pattern, text) if date_pattern else None
    print(text, "->", match.group(0) if match else None)


Date: 21-08-2026 -> 21-08-20
Date: 21/08/2026 -> 21/08/20
Date: 21-08-26 -> 21-08-26
Invalid-looking date: 99-99-2026 -> 99-99-20


## Part 2 — From matching to structured information
**Suggested time: 45–70 min**

Consider the amount forms:

- `₹2450`
- `₹ 2,450`
- `Rs.2450`
- `Rs. 2,450`
- `INR 2450`

### Activity 2.1 — Build an amount extractor
Design a pattern that captures:

1. a currency marker;
2. the numeric amount.

Use **capture groups** so you can inspect the pieces separately.

### Activity 2.2 — Normalize the captured value
Convert:

`Rs. 2,450`

into a structured representation such as:

```python
{"currency": "INR", "amount": 2450}
```

### Think
Why is `2450` more useful computationally than the string `"Rs. 2,450"`?

What information might you still want to retain from the raw text?


In [3]:

amount_tests = [
    "Paid ₹2450 today",
    "Fee: ₹ 2,450",
    "Paid Rs.2450",
    "Paid Rs. 2,450",
    "Paid INR 2450",
]

# TODO 2.1: write an amount pattern with capture groups.
amount_pattern = r"(₹|Rs\.|INR)\s*([0-9,]+)"

for text in amount_tests:
    m = re.search(amount_pattern, text) if amount_pattern else None
    if m:
        print("TEXT:", text)
        print(" full match:", m.group(0))
        print(" groups:", m.groups())
    else:
        print("NO MATCH:", text)


# TODO 2.2: write a function that converts a matched amount to structured data.
def normalize_amount(match):
    # return {"currency": ..., "amount": ...}
    pass


TEXT: Paid ₹2450 today
 full match: ₹2450
 groups: ('₹', '2450')
TEXT: Fee: ₹ 2,450
 full match: ₹ 2,450
 groups: ('₹', '2,450')
TEXT: Paid Rs.2450
 full match: Rs.2450
 groups: ('Rs.', '2450')
TEXT: Paid Rs. 2,450
 full match: Rs. 2,450
 groups: ('Rs.', '2,450')
TEXT: Paid INR 2450
 full match: INR 2450
 groups: ('INR', '2450')


## Part 3 — Break your own regex
**Suggested time: 70–90 min**

A robust NLP workflow does not stop after the first successful match.

### Activity 3.1 — Test difficult cases

| Input | Should your system extract an amount? | Actual result | Correct? |
|---|---:|---|---:|
| `₹2,450` | Yes |  |  |
| `Rs.2450` | Yes |  |  |
| `INR 2450` | Yes |  |  |
| `₹2,450.50` | Yes |  |  |
| `₹2.5k` | Maybe, depending on specification |  |  |
| `two thousand rupees` | Yes semantically; difficult for simple regex |  |  |

### Activity 3.2 — False positive / false negative
Find:

- **one false positive** produced by one of your regexes;
- **one false negative** produced by one of your regexes.

### Required reflection
Complete:

> My regex assumes that ____________________.  
> It fails when ____________________.  
> I would improve it by ____________________.

### Key conceptual test
Does matching `99-99-2026` prove that it is a valid date? Explain.


In [4]:

hard_amounts = [
    "₹2,450",
    "Rs.2450",
    "INR 2450",
    "₹2,450.50",
    "₹2.5k",
    "two thousand rupees",
]

# TODO 3.1: test your current amount_pattern on all cases.
for text in hard_amounts:
    m = re.search(amount_pattern, text) if amount_pattern else None
    print(text, "->", m.group(0) if m else None)

# TODO 3.2: add at least one deliberately difficult input of your own.
my_break_case = ""
print("My break case:", my_break_case)


₹2,450 -> ₹2,450
Rs.2450 -> Rs.2450
INR 2450 -> INR 2450
₹2,450.50 -> ₹2,450
₹2.5k -> ₹2
two thousand rupees -> None
My break case: 


---

## 🛑 STOP POINT 1 — Instructor checkpoint

Before continuing, show or discuss:

1. one regex that works on your intended format;
2. one false positive or false negative;
3. one assumption made by your regex.

**Do not continue until the instructor releases the next part.**

---
